In [1]:
## forward slash always
f_p = 'C:/Users/SAMUEL N-AMOABENG/anaconda3/envs/admission-dec/Yue - Admit Project/admission_assistant'

In [2]:
import os
import pandas as pd
import numpy
import re


def read_reviewer_folder1(folder_path):
    all_files = os.listdir(folder_path)
    file_list = []
    for i, file_name in enumerate(all_files):
        if file_name.endswith('.xlsx'):
            try:
                file_path = os.path.join(folder_path, file_name)
                r1 = pd.read_excel(file_path, header=0)
                r2 = pd.read_excel(file_path, header=1)
                if 'Rating' in r1.columns and 'Name' in r2.columns and 'Institution 1 GPA (4.0 Scale)' in r2.columns:
                    rating = r1['Rating'][1:].to_numpy()
                    name = r2['Name'].to_numpy()
                    gpa = r2['Institution 1 GPA (4.0 Scale)'].astype(str).str.extract(r'(\d+\.\d+)', expand=False).astype(float).to_numpy()
                    splits = file_name.split(' ')
                    round_num = splits[1].split('_')[0] if len(splits) > 1 else ""
                    verd = {'Applicant_Name': name,'GPA': gpa, f'Rating': rating}
                    verdict = pd.DataFrame.from_dict(verd)
                    reviewer_name = re.search(r'_(.+?)\.xlsx', file_name).group(1)
                    verdict['Reviewer_Name'] = reviewer_name
                    file_list.append(verdict)
                else:
                    print(f"File {file_name} does not contain expected columns.")
            except Exception as e:
                print(f"Error reading file {file_name}: {str(e)}")

    return file_list


In [9]:
reviews = read_reviewer_folder1(folder_path=f_p)
reviews

[   Applicant_Name     GPA Rating Reviewer_Name
 0     Bai, Yifang  3.4500      4     COMBINEDi
 1     Bai, Yifang  3.4500      3     COMBINEDi
 2    Cao, Chenrui  3.3416      1     COMBINEDi
 3    Cao, Chenrui  3.3416      1     COMBINEDi
 4       Cao, Hong  3.1600      1     COMBINEDi
 ..            ...     ...    ...           ...
 63  Zheng, Yuxuan  3.2820      4     COMBINEDi
 64  Zheng, Yuxuan  3.2820      3     COMBINEDi
 65     Zhou, Zebo  3.7490      4     COMBINEDi
 66  Zhu, Mingcong  3.3200      1     COMBINEDi
 67  Zhu, Mingcong  3.3200      1     COMBINEDi
 
 [68 rows x 4 columns],
     Applicant_Name     GPA Rating Reviewer_Name
 0    Zhang, Chuzhe  3.0500      3   He_Hangfeng
 1     Han, Chuying  3.3700      4   He_Hangfeng
 2    Zheng, Yuxuan  3.2820      4   He_Hangfeng
 3      Lin, Jingye  3.3000      4   He_Hangfeng
 4   Wang, Jingxing  3.0800      1   He_Hangfeng
 5       Li, Chenyu     NaN      2   He_Hangfeng
 6        Yu, Jiaqi  3.4500      4   He_Hangfeng
 7    

In [4]:
def merge_reviewer_ratings1(file_list):
    # Combine the DataFrames on the Name column
    merged_df = pd.concat(file_list).groupby('Applicant_Name', as_index=False).first()

    # Create a new DataFrame with just the Name column
    name_df = merged_df[['Applicant_Name', 'GPA',]].copy()

    # Loop through the columns for each reviewer's ratings
    for i, df in enumerate(file_list):
        # Rename the Rating column to include the reviewer's name
        df.rename(columns={'Rating': f'reviewer {i+1} Rating'}, inplace=True)

        # Merge the ratings for this reviewer onto the name DataFrame
        name_df = pd.merge(name_df, df[['Applicant_Name', f'reviewer {i+1} Rating']], on='Applicant_Name', how='left')

    return name_df

In [5]:
merge_data =merge_reviewer_ratings1([reviews[1],reviews[2], reviews[3]])
merge_data

,Applicant_Name,GPA,reviewer 1 Rating,reviewer 2 Rating,reviewer 3 Rating
0,"Bai, Yifang",3.4500,4,3,NaN
1,"Cao, Chenrui",3.3416,1,NaN,1
2,"Cao, Hong",3.1600,1,NaN,1
3,"Chen, Jingjing",3.5600,1,NaN,3
4,"Feng, Qiaochu",3.5800,NaN,2,NaN
5,"Feng, Zhihang",3.4300,2,NaN,NaN
6,"Gao, Wanting",3.8900,2,NaN,NaN
7,"Guo, Jiawen",3.2800,1,NaN,3
8,"Han, Chuying",3.3700,4,3,NaN
9,"He, Can",3.8100,NaN,2,NaN


In [27]:
def make_decision_matrix2(decision_df):
    # Initialize decision matrix
    decision_matrix = pd.DataFrame(columns=['Applicant_Name', 'Decision'])

    # Loop through each applicant in the decision dataframe
    for index, row in decision_df.iterrows():
        name = row['Applicant_Name']
        ratings = []
        for col in decision_df.columns:
            if 'Rating' in col:
                rating = row[col]
                if not pd.isna(rating):
                    ratings.append(rating)

        # Get applicant's GPA
        gpa = row['GPA']

        # Check decision conditions
        if not ratings:
            decision = 'Missing 2nd reviewer'
        elif len(ratings) == 1:
            if pd.isna(gpa):
                decision = 'Invalid'
            elif ratings[0] >= 4 and gpa >= 3.6:
                decision = 'ADMIT'
            elif ratings[0] == 3 and gpa >= 3.6:
                decision = 'WAITLIST - HIGH'
            elif ratings[0] == 2 and gpa >= 3.6:
                decision = 'LOOK AGAIN'
            elif ratings[0] == 1 and gpa >= 3.6:
                decision = 'DISCUSS'
            else:
                decision = 'DENY'
        elif len(ratings) == 2:
            ratings.sort()
            rating1, rating2 = ratings
            if rating1 == rating2:
                if rating1 == 1:
                    decision = 'DENY'
                elif rating1 == 2:
                    decision = 'LOOK AGAIN'
                elif rating1 == 3:
                    decision = 'WAITLIST - LOW'
                elif rating1 == 4 or rating1 == 5:
                    decision = 'ADMIT'
            elif rating1 == 1 and rating2 == 2:
                decision = 'LOO AGAIN'
            elif rating1 == 1 and rating2 == 3:
                decision = 'DISCUSS'
            elif rating1 == 1 and rating2 == 4:
                decision = 'DISCUSS'
            elif rating1 == 1 and rating2 == 5:
                decision = 'DISCUSS'
            elif rating1 == 2 and rating2 == 3:
                decision = 'LOOK AGAIN'
            elif rating1 == 2 and rating2 == 4:
                decision = 'LOOK AGAIN'
            elif rating1 == 2 and rating2 == 5:
                decision = 'LOOK AGAIN'
            elif rating1 == 3 and rating2 == 4:
                decision = 'WAITLIST - HIGH'
            elif rating1 == 3 and rating2 == 5:
                decision = 'DISCUSS'
            elif rating1 == 4 and rating2 == 5:
                decision = 'ADMIT'
            else:
                decision = 'DENY'

        # Add decision to decision matrix
        decision_matrix = pd.concat([decision_matrix, pd.DataFrame({'Applicant_Name': [name], 'Decision': [decision]})], ignore_index=True)

    return decision_matrix


In [28]:
applicant_dec = make_decision_matrix2(merge_data)
applicant_dec

,Applicant_Name,Decision
0,"Bai, Yifang",WAITLIST - HIGH
1,"Cao, Chenrui",DENY
2,"Cao, Hong",DENY
3,"Chen, Jingjing",DISCUSS
4,"Feng, Qiaochu",DENY
5,"Feng, Zhihang",DENY
6,"Gao, Wanting",LOOK AGAIN
7,"Guo, Jiawen",DISCUSS
8,"Han, Chuying",WAITLIST - HIGH
9,"He, Can",LOOK AGAIN
